**Tomilin Eugene 12533692**

# Exercise 5 (Spark) [5 points]
---

For this exercise, you will work on this JupyterLab notebook, and solve the tasks listed herein. These tasks, in addition to writing Spark code, require you to analyse various query plans and to reason about them.

To get a deeper understanding, and look up the types and definitions of various functions, we recommend that you visit the Spark and Spark SQL documentation.

### a) From SQL literal syntax to Dataframe (and back again)

In [6]:
import sys
import subprocess

try:
    import pyspark
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pyspark'])

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.storagelevel import StorageLevel

# Create Spark session
spark = (
    SparkSession.builder
    .appName("StackExchangeAnalysis")
    .getOrCreate()
)

# Define schemas for tables used in queries (badges/postlinks use inferSchema only)
comments_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("creationdate", TimestampType(), True),
    StructField("postid", IntegerType(), True),
    StructField("score", IntegerType(), True),
    StructField("text", StringType(), True),
    StructField("userdisplayname", StringType(), True),
    StructField("userid", IntegerType(), True)
])

posts_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("acceptedanswerid", StringType(), True),
    StructField("answercount", IntegerType(), True),
    StructField("body", StringType(), True),
    StructField("closeddate", TimestampType(), True),
    StructField("commentcount", IntegerType(), True),
    StructField("communityowneddate", TimestampType(), True),
    StructField("creationdate", TimestampType(), True),
    StructField("favoritecount", IntegerType(), True),
    StructField("lastactivitydate", TimestampType(), True),
    StructField("lasteditdate", TimestampType(), True),
    StructField("lasteditordisplayname", StringType(), True),
    StructField("lasteditoruserid", StringType(), True),
    StructField("ownerdisplayname", StringType(), True),
    StructField("owneruserid", IntegerType(), True),
    StructField("parentid", IntegerType(), True),
    StructField("posttypeid", StringType(), True),
    StructField("score", IntegerType(), True),
    StructField("tags", StringType(), True),
    StructField("title", StringType(), True),
    StructField("viewcount", IntegerType(), True)
])

users_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("aboutme", StringType(), True),
    StructField("accountid", StringType(), True),
    StructField("creationdate", TimestampType(), True),
    StructField("displayname", StringType(), True),
    StructField("downvotes", IntegerType(), True),
    StructField("lastaccessdate", TimestampType(), True),
    StructField("location", StringType(), True),
    StructField("profileimageurl", StringType(), True),
    StructField("reputation", IntegerType(), True),
    StructField("upvotes", IntegerType(), True),
    StructField("views", IntegerType(), True),
    StructField("websiteurl", StringType(), True)
])

votes_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("bountyamount", IntegerType(), True),
    StructField("creationdate", TimestampType(), True),
    StructField("postid", IntegerType(), True),
    StructField("userid", IntegerType(), True),
    StructField("votetypeid", IntegerType(), True)
])

# bare paths resolve to hdfs://lbddatalab — using file:/// for local FS
base = 'file:///home/adbs_shared/stackexchange'
postsDF = spark.read.options(header=True, inferSchema=True, multiLine=True) \
    .schema(posts_schema).csv(f'{base}/posts.csv')
commentsDF = spark.read.options(header=True, inferSchema=True, multiLine=True) \
    .schema(comments_schema).csv(f'{base}/comments.csv')
usersDF = spark.read.options(header=True, inferSchema=True) \
    .schema(users_schema).csv(f'{base}/users.csv')
votesDF = spark.read.options(header=True, inferSchema=True) \
    .schema(votes_schema).csv(f'{base}/votes.csv')
badgesDF = spark.read.options(header=True, inferSchema=True).csv(f'{base}/badges.csv')
postlinksDF = spark.read.options(header=True, inferSchema=True).csv(f'{base}/postlinks.csv')

# Create temporary views for Spark SQL
badgesDF.createOrReplaceTempView("badges")
commentsDF.createOrReplaceTempView("comments")
postsDF.createOrReplaceTempView("posts")
postlinksDF.createOrReplaceTempView("postlinks")
usersDF.createOrReplaceTempView("users")
votesDF.createOrReplaceTempView("votes")

In [8]:
# Verify schemas loaded correctly (empty DFs on local for un-sampled tables)
for name, df in [("badges", badgesDF), ("comments", commentsDF), ("posts", postsDF),
                  ("postlinks", postlinksDF), ("users", usersDF), ("votes", votesDF)]:
    print(f"=== {name} ===")
    df.printSchema()
    print(f"  rows: {df.count()}")

=== badges ===
root
 |-- id: integer (nullable = true)
 |-- class: integer (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- name: string (nullable = true)
 |-- tagbased: boolean (nullable = true)
 |-- userid: integer (nullable = true)

  rows: 417528
=== comments ===
root
 |-- id: integer (nullable = true)
 |-- creationdate: timestamp (nullable = true)
 |-- postid: integer (nullable = true)
 |-- score: integer (nullable = true)
 |-- text: string (nullable = true)
 |-- userdisplayname: string (nullable = true)
 |-- userid: integer (nullable = true)



  rows: 573838
=== posts ===
root
 |-- id: integer (nullable = true)
 |-- acceptedanswerid: string (nullable = true)
 |-- answercount: integer (nullable = true)
 |-- body: string (nullable = true)
 |-- closeddate: timestamp (nullable = true)
 |-- commentcount: integer (nullable = true)
 |-- communityowneddate: timestamp (nullable = true)
 |-- creationdate: timestamp (nullable = true)
 |-- favoritecount: integer (nullable = true)
 |-- lastactivitydate: timestamp (nullable = true)
 |-- lasteditdate: timestamp (nullable = true)
 |-- lasteditordisplayname: string (nullable = true)
 |-- lasteditoruserid: string (nullable = true)
 |-- ownerdisplayname: string (nullable = true)
 |-- owneruserid: integer (nullable = true)
 |-- parentid: integer (nullable = true)
 |-- posttypeid: string (nullable = true)
 |-- score: integer (nullable = true)
 |-- tags: string (nullable = true)
 |-- title: string (nullable = true)
 |-- viewcount: integer (nullable = true)



  rows: 3169695
=== postlinks ===
root
 |-- id: integer (nullable = true)
 |-- creationdate: timestamp (nullable = true)
 |-- linktypeid: integer (nullable = true)
 |-- postid: integer (nullable = true)
 |-- relatedpostid: integer (nullable = true)

  rows: 67693
=== users ===
root
 |-- id: integer (nullable = true)
 |-- aboutme: string (nullable = true)
 |-- accountid: string (nullable = true)
 |-- creationdate: timestamp (nullable = true)
 |-- displayname: string (nullable = true)
 |-- downvotes: integer (nullable = true)
 |-- lastaccessdate: timestamp (nullable = true)
 |-- location: string (nullable = true)
 |-- profileimageurl: string (nullable = true)
 |-- reputation: integer (nullable = true)
 |-- upvotes: integer (nullable = true)
 |-- views: integer (nullable = true)
 |-- websiteurl: string (nullable = true)



  rows: 322636
=== votes ===
root
 |-- id: integer (nullable = true)
 |-- bountyamount: integer (nullable = true)
 |-- creationdate: timestamp (nullable = true)
 |-- postid: integer (nullable = true)
 |-- userid: integer (nullable = true)
 |-- votetypeid: integer (nullable = true)

  rows: 1360926


#### Query 1: SQL Literal Syntax --> Dataframe API

In [9]:
# votetypeid=1 = AcceptedByOriginator
vt_accept = '1'

query = f"""
WITH answers AS (
  SELECT p.owneruserid AS user_id, COUNT(v.postid) AS accepted_votes
  FROM posts p
  JOIN votes v ON p.id = v.postid
  WHERE p.posttypeid = '2' AND v.votetypeid = '{vt_accept}'
  GROUP BY p.owneruserid
),
questions AS (
  SELECT p.owneruserid AS user_id, COUNT(v.postid) AS upvotes
  FROM posts p
  JOIN votes v ON p.id = v.postid
  WHERE p.posttypeid = '1' AND v.votetypeid = '2'
  GROUP BY p.owneruserid
)
SELECT a.user_id, a.accepted_votes, q.upvotes
FROM answers a
JOIN questions q ON a.user_id = q.user_id
WHERE a.accepted_votes > q.upvotes;
"""

query1 = spark.sql(query)
query1.show()
# query1.explain()

+-------+--------------+-------+
|user_id|accepted_votes|upvotes|
+-------+--------------+-------+
|   7828|            19|      9|
|  28746|            11|      1|
|   8013|            18|      2|
|    686|            34|     11|
| 111259|             4|      1|
|  38185|             3|      1|
|  95569|             4|      3|
|  28732|             3|      1|
|  66341|             3|      1|
| 116195|             4|      2|
|  99274|             2|      1|
|  11887|             6|      1|
+-------+--------------+-------+



In [10]:
# Query 1 — DataFrame API equivalent of the SQL above
vt_accept = '1'  # AcceptedByOriginator

# CTE "answers": posts (type=2) joined with votes (accept-type), count per answer owner
answers = (
    postsDF.alias("p").filter(col("p.posttypeid") == '2')
    .join(votesDF.alias("v").filter(col("v.votetypeid") == vt_accept),
          col("p.id") == col("v.postid"))
    .groupBy(col("p.owneruserid").alias("user_id"))
    .agg(count("v.postid").alias("accepted_votes"))
)
# CTE "questions": posts (type=1) joined with votes (type=2), count upvotes per question owner
questions = (
    postsDF.alias("p").filter(col("p.posttypeid") == '1')
    .join(votesDF.alias("v").filter(col("v.votetypeid") == '2'),
          col("p.id") == col("v.postid"))
    .groupBy(col("p.owneruserid").alias("user_id"))
    .agg(count("v.postid").alias("upvotes"))
)
# Final: inner join on user_id, keep only rows where accepted_votes > upvotes
query1_df = (
    answers.join(questions, "user_id")
    .filter(col("accepted_votes") > col("upvotes"))
    .select("user_id", "accepted_votes", "upvotes")
)
query1_df.show()
# query1_df.explain()

+-------+--------------+-------+
|user_id|accepted_votes|upvotes|
+-------+--------------+-------+
|   7828|            19|      9|
|  28746|            11|      1|
|   8013|            18|      2|
|    686|            34|     11|
| 111259|             4|      1|
|  38185|             3|      1|
|  95569|             4|      3|
|  28732|             3|      1|
|  66341|             3|      1|
| 116195|             4|      2|
|  99274|             2|      1|
|  11887|             6|      1|
+-------+--------------+-------+



#### Query 2: Dataframe API --> SQL Literal Syntax  

In [11]:
min_posts = 50  # threshold on cluster

user_stats = postsDF.filter(col("owneruserid").isNotNull()).groupBy("owneruserid").agg(
                      count("*").alias("post_count"),
                      sum("score").alias("total_score")
                  )

filtered = user_stats.filter(col("total_score").isNotNull() & 
                             (col("post_count") >= min_posts))

result = filtered.withColumn(
    "efficiency",
    col("total_score") / col("post_count")
).orderBy(col("efficiency").desc()).limit(50)

result.show()
#result.explain()

[Stage 98:>                                                         (0 + 1) / 1]

+-----------+----------+-----------+-------------------+
|owneruserid|post_count|total_score|         efficiency|
+-----------+----------+-----------+-------------------+
|       4253|        52|        362|  6.961538461538462|
|      14076|        51|        266|  5.215686274509804|
|      53690|        51|        176|  3.450980392156863|
|          3|       101|        345| 3.4158415841584158|
|          5|        58|        191|  3.293103448275862|
|        805|       114|        341|  2.991228070175439|
|      11032|        98|        263|  2.683673469387755|
|        686|       159|        396|  2.490566037735849|
|          2|        69|        141| 2.0434782608695654|
|      36041|        54|        110|  2.037037037037037|
|          0|       323|        511| 1.5820433436532508|
|       7828|        67|         79| 1.1791044776119404|
|      11887|        69|         81|  1.173913043478261|
|          1|       209|        231|  1.105263157894737|
|          4|       329|       

In [12]:
# Query 2 — SQL literal syntax equivalent of the DataFrame API above
min_posts = 50  # cluster threshold

query2_sql = f"""
SELECT owneruserid, post_count, total_score,
       total_score / post_count AS efficiency
FROM (
  SELECT owneruserid,
         COUNT(*) AS post_count,
         SUM(score) AS total_score
  FROM posts
  WHERE owneruserid IS NOT NULL
  GROUP BY owneruserid
) t
WHERE total_score IS NOT NULL
  AND post_count >= {min_posts}
ORDER BY efficiency DESC
LIMIT 50
"""
query2 = spark.sql(query2_sql)
query2.show()
# query2.explain()

[Stage 101:>                                                        (0 + 1) / 1]

+-----------+----------+-----------+-------------------+
|owneruserid|post_count|total_score|         efficiency|
+-----------+----------+-----------+-------------------+
|       4253|        52|        362|  6.961538461538462|
|      14076|        51|        266|  5.215686274509804|
|      53690|        51|        176|  3.450980392156863|
|          3|       101|        345| 3.4158415841584158|
|          5|        58|        191|  3.293103448275862|
|        805|       114|        341|  2.991228070175439|
|      11032|        98|        263|  2.683673469387755|
|        686|       159|        396|  2.490566037735849|
|          2|        69|        141| 2.0434782608695654|
|      36041|        54|        110|  2.037037037037037|
|          0|       323|        511| 1.5820433436532508|
|       7828|        67|         79| 1.1791044776119404|
|      11887|        69|         81|  1.173913043478261|
|          1|       209|        231|  1.105263157894737|
|          4|       329|       

### b) SPARK processing model
You are given two independent Spark applications. Each application consists of a sequence of Spark jobs, and each job consists of multiple transformations followed by actions. Two optimization blocks have been introduced in an attempt to improve the performance of these applications.<br>
Based on the provided programs, answer the following questions:
* Ignoring the two optimization blocks, identify the stages and their boundaries for each Spark job: *user_scores*, *tag_scores*, and *user_comment_scores*.
Clearly explain your reasoning using Spark’s processing model.
* For Application 1, analyze and discuss the impact of the two optimization blocks on the performance of the application. Explain whether they improve performance, and justify your answer.
 * For Application 2, analyze and discuss the impact of the two optimization blocks on the performance of the application. Again, explain whether they improve performance, providing justification.


In [13]:
posts = postsDF.select('id','owneruserid','tags','score').rdd 
posts = posts.filter(lambda x: (x['score'] != None) & (x['owneruserid'] != None) & (x['tags'] != None)).map(lambda x: (x['id'], (x['owneruserid'], x['tags'], int(x['score']))))# (post_id, (user_id, tag, score))

votes = votesDF.rdd
votes = votes.map(lambda x: (x['postid'], 1)) # (post_id, vote_count)

comments = commentsDF.rdd
comments = comments.map(lambda x: (x['postid'], 1)) # (post_id, comment_count)

high_score_posts = posts.filter(lambda x: x[1][2] >= 5)

# ---------------------------
# OPTIMIZATION BLOCK 1
# ---------------------------
high_score_posts = high_score_posts.partitionBy(4).cache()
votes = votes.partitionBy(4)
#-----------------------------

post_votes = high_score_posts.join(votes)
post_comments = high_score_posts.join(comments)

# ---------------------------
# OPTIMIZATION BLOCK 2
# ---------------------------
post_votes.cache()
#-----------------------------

user_scores = post_votes.map(lambda x: (x[1][0][0], x[1][1])) \
                        .reduceByKey(lambda a, b: a + b)

tag_scores = post_votes.map(lambda x: (x[1][0][1], x[1][1])) \
                      .reduceByKey(lambda a, b: a + b)

user_comment_scores = post_comments.map(lambda x: (x[1][0][0], x[1][1])) \
                        .reduceByKey(lambda a, b: a + b)


In [14]:
# ---------------------------
# Application 1
# ---------------------------
print(user_scores.take(5))
print(tag_scores.take(5))
print(user_comment_scores.take(5))

[(196, 79), (168, 21), (2164, 6), (4096, 7), (2728, 10)]
[('<time-series><mcmc><mixed-model><forecasting>', 10), ('<r><anova><contrasts><sums-of-squares>', 39), ('<econometrics>', 22), ('<mixed-model>', 6), ('<interpretation><moments><parallel-computing>', 22)]


[Stage 114:=============================================>           (4 + 1) / 5]

[(15, 55974), (0, 22390), (45, 7462), (40, 14924), (1500, 3731)]


In [15]:
# ---------------------------
# Application 2
# ---------------------------
print(user_scores.take(5))
print(tag_scores.take(5))

[(196, 79), (168, 21), (2164, 6), (4096, 7), (2728, 10)]
[('<time-series><mcmc><mixed-model><forecasting>', 10), ('<r><anova><contrasts><sums-of-squares>', 39), ('<econometrics>', 22), ('<mixed-model>', 6), ('<interpretation><moments><parallel-computing>', 22)]


### b) Analysis — Stages and Optimization Blocks

#### Approach used

I've ran 4-way comparison (see bottom cell) to get lineage dumps and wall-clock times for each OB combination. Stage boundaries are visible as `ShuffledRDD` entries in the lineage string. Mermaids and comparison table below illustrates my findings. Surprisingly 1st caching hasn't benefited much.

#### Measured results (cluster run, all 3 jobs actioned)

| Config | Wall time | vs baseline | Lineage notes |
|---|---|---|---|
| OB1- OB2- | 136.8s | — | Two join shuffles, no caching |
| OB1- OB2+ | 134.7s | -1.5% | `CachedPartitions` on post_votes visible, but join shuffle still dominates |
| OB1+ OB2- | 82.3s | -40% | `PartitionerAwareUnionRDD` replaces join shuffle; `CachedPartitions` on high_score_posts |
| OB1+ OB2+ | 79.0s | -42% | Co-partitioned join + both caches active |

#### Plan figures

**OB1- OB2- (baseline, 136.8s)**

```mermaid
graph TD
    P["postsDF: filter nulls, map to (id, (uid,tag,score))"]
    V["votesDF: map to (postid, 1)"]
    C["commentsDF: map to (postid, 1)"]
    HP["high_score_posts: filter score>=5"]
    J1["join(posts, votes) -- Shuffle on post_id"]
    J2["join(posts, comments) -- Shuffle on post_id"]
    R1["reduceByKey(user_id) -- Shuffle on user_id"]
    R2["reduceByKey(tag) -- Shuffle on tag"]
    R3["reduceByKey(user_id) -- Shuffle on user_id"]
    US(["user_scores"])
    TS(["tag_scores"])
    UCS(["user_comment_scores"])

    P --> HP
    V --> J1
    C --> J2
    HP --> J1
    HP --> J2
    J1 --> R1
    J1 --> R2
    J2 --> R3
    R1 --> US
    R2 --> TS
    R3 --> UCS
```

**OB1- OB2+ (134.7s, -1.5%)**

```mermaid
graph TD
    P["postsDF: filter nulls, map to (id, (uid,tag,score))"]
    V["votesDF: map to (postid, 1)"]
    C["commentsDF: map to (postid, 1)"]
    HP["high_score_posts: filter score>=5"]
    J1["join(posts, votes) -- Shuffle on post_id"]
    J2["join(posts, comments) -- Shuffle on post_id"]
    PV["post_votes .cache() -- CachedPartitions: 4"]
    R1["reduceByKey(user_id)"]
    R2["reduceByKey(tag)"]
    R3["reduceByKey(user_id)"]
    US(["user_scores"])
    TS(["tag_scores"])
    UCS(["user_comment_scores"])

    P --> HP
    V --> J1
    C --> J2
    HP --> J1
    HP --> J2
    J1 --> PV
    J2 --> R3
    PV --> R1
    PV --> R2
    R1 --> US
    R2 --> TS
    R3 --> UCS

    style PV fill:#e6ffe6,stroke:#4caf50
```

**OB1+ OB2- (82.3s, -40%)**

```mermaid
graph TD
    P["postsDF: filter nulls, map to (id, (uid,tag,score))"]
    V["votesDF: map to (postid, 1)"]
    C["commentsDF: map to (postid, 1)"]
    HP["high_score_posts .partitionBy(4).cache()"]
    VP["votes .partitionBy(4)"]
    J1["join(posts, votes) -- NARROW, co-partitioned"]
    J2["join(posts, comments) -- Shuffle on post_id"]
    R1["reduceByKey(user_id) -- Shuffle on user_id"]
    R2["reduceByKey(tag) -- Shuffle on tag"]
    R3["reduceByKey(user_id) -- Shuffle on user_id"]
    US(["user_scores"])
    TS(["tag_scores"])
    UCS(["user_comment_scores"])

    P --> HP
    V --> VP
    C --> J2
    HP --> J1
    HP --> J2
    VP --> J1
    J1 --> R1
    J1 --> R2
    J2 --> R3
    R1 --> US
    R2 --> TS
    R3 --> UCS

    style HP fill:#e6ffe6,stroke:#4caf50
    style J1 fill:#fff3e0,stroke:#ff9800
```

**OB1+ OB2+ (79.0s, -42%)**

```mermaid
graph TD
    P["postsDF: filter nulls, map to (id, (uid,tag,score))"]
    V["votesDF: map to (postid, 1)"]
    C["commentsDF: map to (postid, 1)"]
    HP["high_score_posts .partitionBy(4).cache()"]
    VP["votes .partitionBy(4)"]
    J1["join(posts, votes) -- NARROW, co-partitioned"]
    J2["join(posts, comments) -- Shuffle on post_id"]
    PV["post_votes .cache() -- CachedPartitions: 4"]
    R1["reduceByKey(user_id)"]
    R2["reduceByKey(tag)"]
    R3["reduceByKey(user_id)"]
    US(["user_scores"])
    TS(["tag_scores"])
    UCS(["user_comment_scores"])

    P --> HP
    V --> VP
    C --> J2
    HP --> J1
    HP --> J2
    VP --> J1
    J1 --> PV
    J2 --> R3
    PV --> R1
    PV --> R2
    R1 --> US
    R2 --> TS
    R3 --> UCS

    style HP fill:#e6ffe6,stroke:#4caf50
    style PV fill:#e6ffe6,stroke:#4caf50
    style J1 fill:#fff3e0,stroke:#ff9800
```

#### Observations
Without OB1, `high_score_posts` has no partitioner and isn't cached. It is **recomputed from scratch for each join** (once for votes, once for comments). Without OB2, `post_votes` is **recomputed twice** — once when `user_scores` is actioned, once for `tag_scores` — both trace lineage back through the join.

##### OB1 effects `[high_score_posts.partitionBy(4).cache()` + `votes.partitionBy(4)]`

The lineage confirms this: with OB1 on, the join produces a `PartitionerAwareUnionRDD` instead of a second `ShuffledRDD`. The `partitionBy(4)` shuffle is a one-time cost. After that, both RDDs share the same `HashPartitioner` on `post_id`, so the join needs no extra shuffle.

`.cache()` on `high_score_posts` is confirmed by `CachedPartitions: 4; MemorySize: 26.4 KiB` in the lineage dump. It gets reused across both joins (votes and comments).

**OB1 conclusion: clear win.** 40% speedup. The partitionBy shuffle pays for itself by eliminating the join shuffle — and caching prevents recomputation.

##### OB2 effects [post_votes.cache()]

Without OB1: OB2 alone saves only 2.1s (1.5%). The lineage shows `CachedPartitions: 4; MemorySize: 34.6 KiB` on `post_votes`, confirming it's cached — but the dominant cost is still the two join shuffles (posts+votes and posts+comments), which caching `post_votes` does nothing about.

With OB1 (join shuffle already eliminated): OB2 trims another 3.3s (from 82.3s to 79.0s, about 4%). The cached `post_votes` avoids re-running the co-partitioned join for the second `reduceByKey`.

**OB2 conclusion: useless alone, small additive benefit with OB1.** The `take(5)` action only pulls a tiny fraction of data, so the cost of recomputing `post_votes` from the co-partitioned join is low. OB2 would matter more for full `.collect()` or iterative workloads.

#### Effects on Application 1 vs Application 2

App 1 actions all three jobs. App 2 actions only `user_scores` and `tag_scores` — `user_comment_scores` is not evaluated.

| Block | App 1 (all 3 jobs) | App 2 (2 jobs, comments pruned) |
|---|---|---|
| OB1 | **Helps (40%).** Two joins share cached `high_score_posts`, co-partitioning eliminates one shuffle. | **Neutral to slight loss.** Only one join uses `high_score_posts`, so the cache is wasted. partitionBy shuffle replaces the join shuffle — cost is roughly a wash. |
| OB2 | **Marginal alone, helps with OB1.** Without OB1, caching `post_votes` barely matters because join shuffles dominate. With OB1, saves 4%. | **Helps.** `post_votes` still consumed by two `reduceByKey` ops; caching avoids recomputing the join. |

#### Summary

OB1 contributes most — co-partitioning eliminates a full shuffle, while OB2 is only meaningful once the join shuffle is already gone. For App 2 where the comments branch is pruned, OB1's cache is wasted so its benefit shrinks to near zero.

### c) Translate the algorithm for constructing a Top-K user similarity graph from a user-item interaction graph (algorithm from Exercise 2)

In [16]:
# Input edges
edges = [
    ("u1", "i1"), ("u1", "i2"), ("u1", "i3"), ("u1", "i4"),
    ("u2", "i1"), ("u2", "i2"), ("u2", "i5"),
    ("u3", "i2"), ("u3", "i3"), ("u3", "i4"),
    ("u4", "i3"), ("u4", "i5")
]

k = 2 

In [17]:
# Step 1 — reverse edges: (item, user)
edges_rdd = spark.sparkContext.parallelize(edges)
item_users = edges_rdd.map(lambda e: (e[1], e[0]))  # (item, user)

# Step 2 — group users by item, generate all unordered user pairs per item
def _pairs_per_item(kv):  # kv is (item, [users]) from groupByKey
    users = list(set(kv[1]))  # dedup in case of duplicate edges
    out = []
    for i in range(len(users)):
        for j in range(i + 1, len(users)):
            u1, u2 = sorted([users[i], users[j]])  # canonical ordering
            out.append(((u1, u2), 1))
    return out

pair_counts = item_users.groupByKey().flatMap(_pairs_per_item)  # ((u1,u2), 1)

# Step 3 — sum co-occurrence counts per user pair
similarity = pair_counts.reduceByKey(lambda a, b: a + b)  # ((u1,u2), sim)

# Step 4 — duplicate for each user, group by user, keep top-K
def _expand_per_user(kv):  # kv is ((u1,u2), sim)
    (u1, u2), sim = kv
    return [(u1, (u2, sim)), (u2, (u1, sim))]

user_sims = similarity.flatMap(_expand_per_user)  # (user, (other, sim))

def _topk_for_user(kv, k=k):  # kv is (user, [(other, sim), ...])
    sims = sorted(kv[1], key=lambda x: -x[1])[:k]  # desc by similarity
    return [(kv[0], other, s) for other, s in sims]

topk_graph = user_sims.groupByKey().flatMap(_topk_for_user)

print("Top-K User Similarity Graph (k=2):")
for row in topk_graph.collect():
    print(f"  {row[0]} → {row[1]} (sim={row[2]})")

Top-K User Similarity Graph (k=2):
  u4 → u3 (sim=1)
  u4 → u1 (sim=1)
  u2 → u1 (sim=2)
  u2 → u3 (sim=1)
  u1 → u3 (sim=3)
  u1 → u2 (sim=2)
  u3 → u1 (sim=3)
  u3 → u2 (sim=1)


### d) You are given a dataset of retail transactions, where each transaction consists of a list of items bought together. Using Spark RDD transformations and actions, implement an algorithm to identify the top-K most frequently co-purchased item pairs.

In [18]:
# Input Data
transactions = [
    ["milk", "bread", "eggs", "butter"],
    ["bread", "butter", "jam"],
    ["milk", "bread"],
    ["bread", "eggs"],
    ["milk", "eggs", "yogurt"],
    ["bread", "milk", "butter"],
    ["milk", "bread", "butter", "cheese"],
    ["bread", "eggs", "butter"],
    ["milk", "eggs", "bread"],
    ["bread", "butter", "jam", "honey"],
    ["milk", "yogurt"],
    ["eggs", "bread", "milk", "butter"],
    ["bread", "cheese"],
    ["milk", "bread", "eggs"],
    ["butter", "bread", "jam"],
    ["milk", "bread", "cheese"],
    ["eggs", "milk", "yogurt", "honey"],
    ["bread", "butter", "eggs"],
    ["milk", "bread", "butter", "jam"],
    ["bread", "eggs", "milk"]
]

min_support = 3

In [19]:
# Identify the top-K most frequently co-purchased item pairs
# K not specified in task — output all pairs meeting min_support, sorted by count desc
trans_rdd = spark.sparkContext.parallelize(transactions)

# Generate all unordered item pairs per transaction
def _pairs_from_basket(basket):
    distinct = list(set(basket))  # dedup within a basket
    out = []
    for i in range(len(distinct)):
        for j in range(i + 1, len(distinct)):
            pair = tuple(sorted([distinct[i], distinct[j]]))  # canonical order
            out.append((pair, 1))
    return out

pair_counts = trans_rdd.flatMap(_pairs_from_basket).reduceByKey(lambda a, b: a + b)

# Filter by minimum support threshold
frequent = pair_counts.filter(lambda x: x[1] >= min_support)

# Sort descending by count, collect all (top-K = all above support)
result = frequent.sortBy(lambda x: -x[1]).collect()

print("Co-purchased item pairs (min_support=3):")
for (i1, i2), cnt in result:
    print(f"  {i1} + {i2} → {cnt}")

Co-purchased item pairs (min_support=3):
  bread + butter → 10
  bread + milk → 10
  bread + eggs → 8
  eggs + milk → 7
  butter + milk → 5
  butter + eggs → 4
  butter + jam → 4
  bread + jam → 4
  milk + yogurt → 3
  bread + cheese → 3


---
## **Your solution for Exercise 5 will consist of:**  
*  This notebook, filled with your solutions and discussions/explanations in the report. 


In [ ]:
# Compare all 4 OB1/OB2 combinations: RDD lineage + execution time
import time

def run_app(use_ob1, use_ob2, label):
    p = postsDF.select('id','owneruserid','tags','score').rdd
    p = p.filter(lambda x: (x['score'] is not None) & (x['owneruserid'] is not None) & (x['tags'] is not None)) \
         .map(lambda x: (x['id'], (x['owneruserid'], x['tags'], int(x['score']))))

    v = votesDF.rdd.map(lambda x: (x['postid'], 1))
    c = commentsDF.rdd.map(lambda x: (x['postid'], 1))

    hp = p.filter(lambda x: x[1][2] >= 5)

    if use_ob1:
        hp = hp.partitionBy(4).cache()
        v = v.partitionBy(4)

    pv = hp.join(v)
    pc = hp.join(c)

    if use_ob2:
        pv = pv.cache()

    us = pv.map(lambda x: (x[1][0][0], x[1][1])).reduceByKey(lambda a, b: a + b)
    ts = pv.map(lambda x: (x[1][0][1], x[1][1])).reduceByKey(lambda a, b: a + b)
    ucs = pc.map(lambda x: (x[1][0][0], x[1][1])).reduceByKey(lambda a, b: a + b)

    t0 = time.time()
    r1 = us.take(5)  # triggers the whole DAG
    r2 = ts.take(5)  # reuse cached post_votes if OB2
    r3 = ucs.take(5)
    elapsed = time.time() - t0

    print(f"--- {label} ---")
    print(f"  time: {elapsed:.3f}s")
    print(f"  user_scores:  {r1}")
    print(f"  tag_scores:   {r2}")
    print(f"  ucs:          {r3}")
    print()

    print("Lineage (us):")
    print(us.toDebugString())
    print()

print("=== OB1- OB2- (baseline) ===")
run_app(False, False, "OB1- OB2-")

print("=== OB1- OB2+ ===")
run_app(False, True,  "OB1- OB2+")

print("=== OB1+ OB2- ===")
run_app(True,  False, "OB1+ OB2-")

print("=== OB1+ OB2+ (both) ===")
run_app(True,  True,  "OB1+ OB2+")

=== OB1- OB2- (baseline) ===
                                                                                
--- OB1- OB2- ---
  time: 136.810s
  user_scores:  [(196, 79), (168, 21), (2164, 6), (4096, 7), (2728, 10)]
  tag_scores:   [('<time-series><mcmc><mixed-model><forecasting>', 10), ('<r><anova><contrasts><sums-of-squares>', 39), ('<econometrics>', 22), ('<mixed-model>', 6), ('<interpretation><moments><parallel-computing>', 22)]
  ucs:          [(104, 3), (2, 26119), (24, 14924), (28, 3731), (0, 22390)]

Lineage (us):
(4) PythonRDD[385] at RDD at PythonRDD.scala:53 []
 |  MapPartitionsRDD[373] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[372] at partitionBy at <unknown>:0 []
 +-(4) PairwiseRDD[371] at reduceByKey at /tmp/ipykernel_8903/650388776.py:24 []
    |  PythonRDD[370] at reduceByKey at /tmp/ipykernel_8903/650388776.py:24 []
    |  MapPartitionsRDD[362] at mapPartitions at PythonRDD.scala:160 []
    |  ShuffledRDD[361] at partitionBy at <unknown>:0 []
    +-(4) PairwiseRDD[360] at join at /tmp/ipykernel_8903/650388776.py:18 []
       |  PythonRDD[359] at join at /tmp/ipykernel_8903/650388776.py:18 []
       |  UnionRDD[358] at union at NativeMethodAccessorImpl.java:0 []
       |  PythonRDD[356] at RDD at PythonRDD.scala:53 []
       |  MapPartitionsRDD[355] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  MapPartitionsRDD[354] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  SQLExecutionRDD[353] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  MapPartitionsRDD[352] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  MapPartitionsRDD[351] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  FileScanRDD[350] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  PythonRDD[357] at RDD at PythonRDD.scala:53 []
       |  MapPartitionsRDD[246] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  MapPartitionsRDD[245] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  SQLExecutionRDD[244] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  MapPartitionsRDD[243] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  FileScanRDD[242] at javaToPython at NativeMethodAccessorImpl.java:0 []

=== OB1- OB2+ ===
                                                                                
--- OB1- OB2+ ---
  time: 134.679s
  user_scores:  [(196, 79), (168, 21), (2164, 6), (4096, 7), (2728, 10)]
  tag_scores:   [('<time-series><mcmc><mixed-model><forecasting>', 10), ('<r><anova><contrasts><sums-of-squares>', 39), ('<econometrics>', 22), ('<mixed-model>', 6), ('<interpretation><moments><parallel-computing>', 22)]
  ucs:          [(104, 3), (2, 26119), (24, 14924), (28, 3731), (0, 22390)]

Lineage (us):
(4) PythonRDD[422] at RDD at PythonRDD.scala:53 []
 |  MapPartitionsRDD[410] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[409] at partitionBy at <unknown>:0 []
 +-(4) PairwiseRDD[408] at reduceByKey at /tmp/ipykernel_8903/650388776.py:24 []
    |  PythonRDD[407] at reduceByKey at /tmp/ipykernel_8903/650388776.py:24 []
    |  PythonRDD[406] at RDD at PythonRDD.scala:53 []
    |      CachedPartitions: 4; MemorySize: 34.6 KiB; DiskSize: 0.0 B
    |  MapPartitionsRDD[398] at mapPartitions at PythonRDD.scala:160 []
    |  ShuffledRDD[397] at partitionBy at <unknown>:0 []
    +-(4) PairwiseRDD[396] at join at /tmp/ipykernel_8903/650388776.py:18 []
       |  PythonRDD[395] at join at /tmp/ipykernel_8903/650388776.py:18 []
       |  UnionRDD[394] at union at NativeMethodAccessorImpl.java:0 []
       |  PythonRDD[392] at RDD at PythonRDD.scala:53 []
       |  MapPartitionsRDD[391] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  MapPartitionsRDD[390] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  SQLExecutionRDD[389] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  MapPartitionsRDD[388] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  MapPartitionsRDD[387] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  FileScanRDD[386] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  PythonRDD[393] at RDD at PythonRDD.scala:53 []
       |  MapPartitionsRDD[246] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  MapPartitionsRDD[245] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  SQLExecutionRDD[244] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  MapPartitionsRDD[243] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  FileScanRDD[242] at javaToPython at NativeMethodAccessorImpl.java:0 []

=== OB1+ OB2- ===
                                                                                
--- OB1+ OB2- ---
  time: 82.333s
  user_scores:  [(196, 79), (168, 21), (2164, 6), (4096, 7), (2728, 10)]
  tag_scores:   [('<time-series><mcmc><mixed-model><forecasting>', 10), ('<r><anova><contrasts><sums-of-squares>', 39), ('<econometrics>', 22), ('<mixed-model>', 6), ('<interpretation><moments><parallel-computing>', 22)]
  ucs:          [(15, 55974), (0, 22390), (45, 7462), (40, 14924), (1500, 3731)]

Lineage (us):
(4) PythonRDD[462] at RDD at PythonRDD.scala:53 []
 |  MapPartitionsRDD[450] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[449] at partitionBy at <unknown>:0 []
 +-(4) PairwiseRDD[448] at reduceByKey at /tmp/ipykernel_8903/650388776.py:24 []
    |  PythonRDD[447] at reduceByKey at /tmp/ipykernel_8903/650388776.py:24 []
    |  PartitionerAwareUnionRDD[439] at union at NativeMethodAccessorImpl.java:0 []
    |  PythonRDD[437] at RDD at PythonRDD.scala:53 []
    |  MapPartitionsRDD[432] at mapPartitions at PythonRDD.scala:160 []
    |      CachedPartitions: 4; MemorySize: 26.4 KiB; DiskSize: 0.0 B
    |  ShuffledRDD[431] at partitionBy at <unknown>:0 []
    +-(1) PairwiseRDD[430] at partitionBy at /tmp/ipykernel_8903/650388776.py:15 []
       |  PythonRDD[429] at partitionBy at /tmp/ipykernel_8903/650388776.py:15 []
       |  MapPartitionsRDD[428] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  MapPartitionsRDD[427] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  SQLExecutionRDD[426] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  MapPartitionsRDD[425] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  MapPartitionsRDD[424] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  FileScanRDD[423] at javaToPython at NativeMethodAccessorImpl.java:0 []
    |  PythonRDD[438] at RDD at PythonRDD.scala:53 []
    |  MapPartitionsRDD[436] at mapPartitions at PythonRDD.scala:160 []
    |  ShuffledRDD[435] at partitionBy at <unknown>:0 []
    +-(3) PairwiseRDD[434] at partitionBy at /tmp/ipykernel_8903/650388776.py:16 []
       |  PythonRDD[433] at partitionBy at /tmp/ipykernel_8903/650388776.py:16 []
       |  MapPartitionsRDD[246] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  MapPartitionsRDD[245] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  SQLExecutionRDD[244] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  MapPartitionsRDD[243] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  FileScanRDD[242] at javaToPython at NativeMethodAccessorImpl.java:0 []

=== OB1+ OB2+ (both) ===
[Stage 187:=============================================>           (4 + 1) / 5]
--- OB1+ OB2+ ---
  time: 79.044s
  user_scores:  [(196, 79), (168, 21), (2164, 6), (4096, 7), (2728, 10)]
  tag_scores:   [('<time-series><mcmc><mixed-model><forecasting>', 10), ('<r><anova><contrasts><sums-of-squares>', 39), ('<econometrics>', 22), ('<mixed-model>', 6), ('<interpretation><moments><parallel-computing>', 22)]
  ucs:          [(15, 55974), (0, 22390), (45, 7462), (40, 14924), (1500, 3731)]

Lineage (us):
(4) PythonRDD[503] at RDD at PythonRDD.scala:53 []
 |  MapPartitionsRDD[491] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[490] at partitionBy at <unknown>:0 []
 +-(4) PairwiseRDD[489] at reduceByKey at /tmp/ipykernel_8903/650388776.py:24 []
    |  PythonRDD[488] at reduceByKey at /tmp/ipykernel_8903/650388776.py:24 []
    |  PythonRDD[487] at RDD at PythonRDD.scala:53 []
    |      CachedPartitions: 4; MemorySize: 34.6 KiB; DiskSize: 0.0 B
    |  PartitionerAwareUnionRDD[479] at union at NativeMethodAccessorImpl.java:0 []
    |  PythonRDD[477] at RDD at PythonRDD.scala:53 []
    |  MapPartitionsRDD[472] at mapPartitions at PythonRDD.scala:160 []
    |      CachedPartitions: 4; MemorySize: 26.4 KiB; DiskSize: 0.0 B
    |  ShuffledRDD[471] at partitionBy at <unknown>:0 []
    +-(1) PairwiseRDD[470] at partitionBy at /tmp/ipykernel_8903/650388776.py:15 []
       |  PythonRDD[469] at partitionBy at /tmp/ipykernel_8903/650388776.py:15 []
       |  MapPartitionsRDD[468] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  MapPartitionsRDD[467] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  SQLExecutionRDD[466] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  MapPartitionsRDD[465] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  MapPartitionsRDD[464] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  FileScanRDD[463] at javaToPython at NativeMethodAccessorImpl.java:0 []
    |  PythonRDD[478] at RDD at PythonRDD.scala:53 []
    |  MapPartitionsRDD[476] at mapPartitions at PythonRDD.scala:160 []
    |  ShuffledRDD[475] at partitionBy at <unknown>:0 []
    +-(3) PairwiseRDD[474] at partitionBy at /tmp/ipykernel_8903/650388776.py:16 []
       |  PythonRDD[473] at partitionBy at /tmp/ipykernel_8903/650388776.py:16 []
       |  MapPartitionsRDD[246] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  MapPartitionsRDD[245] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  SQLExecutionRDD[244] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  MapPartitionsRDD[243] at javaToPython at NativeMethodAccessorImpl.java:0 []
       |  FileScanRDD[242] at javaToPython at NativeMethodAccessorImpl.java:0 []

   